# MLflow AI Gateway
In this notebook, we'll demonstrate how to use MLflow's AI Gateway. The AI Gateway is similar to LiteLLM in the sense that it can serve out multiple LLM providers behind a single endpoint. It also provides extra goodies like being able to track usage and routing for A/B testing.

## 0: Notebook Setup

In [1]:
# Importing the necessary Python libraries
import anthropic
import requests
from openai import OpenAI

In [2]:
# Setting the base URL for our MLflow instance
MLFLOW_BASE_URL = 'http://127.0.0.1:5000'

# Setting the respective model endpoint names
OPENAI_GPT_5_4_NANO_ENDPOINT = 'openai-gpt-5.4-nano'
CLAUDE_HAIKU_4_5_ENDPOINT = 'anthropic-claude-haiku-4.5-endpoint'

In [3]:
# Creating a demo prompts
DEMO_PROMPT = 'What is the capital of Illinois?'
JAR_JAR_PROMPT = 'What is the capital of Illinois? It would be fun to have the response in the tone of Jar Jar Binks, but ultimately, I just need the answer to my first question.'

## 1: Model Invocation

### 1.1: OpenAI Invocations

#### 1.1.1: Invoking with the OpenAI Python Client

In [4]:
# Instantiating the OpenAI SDK client
openai_client = OpenAI(
    base_url = f'{MLFLOW_BASE_URL}/gateway/openai/v1',
    api_key = 'dummy'
)

# Invoking the OpenAI model with the OpenAI SDK
response = openai_client.chat.completions.create(
    model = OPENAI_GPT_5_4_NANO_ENDPOINT,
    messages=[
        {
            'role': 'user',
            'content': DEMO_PROMPT
        }
    ],
)

# Printing the response
print(response.choices[0].message.content)

The capital of Illinois is **Springfield**.


#### 1.1.2: Invoking with with cURL (through Requests)

In [5]:
# Invoking the OpenAI model with the Requests library
response = requests.post(
    url = f'{MLFLOW_BASE_URL}/gateway/{OPENAI_GPT_5_4_NANO_ENDPOINT}/mlflow/invocations',
    json = {
        'messages': [
            {
                'role': 'user',
                'content': DEMO_PROMPT
            },
        ]
    }
)

# Printing the response
print(response.json()['choices'][0]['message']['content'])

The capital of Illinois is **Springfield**.


#### 1.1.3 Testing Jar Jar Guardrail

In [6]:
# Instantiating the OpenAI SDK client
openai_client = OpenAI(
    base_url = f'{MLFLOW_BASE_URL}/gateway/openai/v1',
    api_key = 'dummy'
)

# Invoking the OpenAI model with the OpenAI SDK
try:
    response = openai_client.chat.completions.create(
        model = OPENAI_GPT_5_4_NANO_ENDPOINT,
        messages=[
            {
                'role': 'user',
                'content': JAR_JAR_PROMPT
            }
        ],
    )

    # Printing the response
    print(response.json()['choices'][0]['message']['content'])
    
except Exception as e:
    print(e)

Error code: 400 - {'detail': "Guardrail 'Jar Jar Guardrail' blocked: The input explicitly requests that the response be provided in the tone of Jar Jar Binks (“It would be fun to have the response in the tone of Jar Jar Binks”). This is directly related to Jar Jar Binks, so it is not safe per the criteria."}


### 1.2: Anthropic Claude Invocation

#### 1.2.1: Invoking with the Anthropic Python Client

In [7]:
# Instantiating the Anthropic SDK client
anthropic_client = anthropic.Anthropic(
    base_url = f'{MLFLOW_BASE_URL}/gateway/anthropic',
    api_key = 'dummy'
)

# Invoking the Anthropic model with the Anthropic SDK
response = anthropic_client.messages.create(
    model = CLAUDE_HAIKU_4_5_ENDPOINT,
    max_tokens = 10000,
    messages=[
        {
            'role': 'user',
            'content': DEMO_PROMPT
        }
    ],
)

# Printing the response
print(response.content[0].text)

The capital of Illinois is Springfield.


#### 1.2.2: Invoking with with cURL (through Requests)

In [8]:
# Invoking the Anthropic Claude model with the Requests library
response = requests.post(
    url = f'{MLFLOW_BASE_URL}/gateway/{CLAUDE_HAIKU_4_5_ENDPOINT}/mlflow/invocations',
    json = {
        'messages': [
            {
                'role': 'user',
                'content': DEMO_PROMPT
            },
        ]
    }
)

# Printing the response
print(response.json()['choices'][0]['message']['content'])

The capital of Illinois is Springfield.


#### 1.2.3 Testing Jar Jar Guardrail

In [9]:
# Instantiating the Anthropic SDK client
anthropic_client = anthropic.Anthropic(
    base_url = f'{MLFLOW_BASE_URL}/gateway/anthropic',
    api_key = 'dummy'
)

# Invoking the Anthropic model with the Anthropic SDK
try:
    response = anthropic_client.messages.create(
        model = CLAUDE_HAIKU_4_5_ENDPOINT,
        max_tokens = 10000,
        messages=[
            {
                'role': 'user',
                'content': JAR_JAR_PROMPT
            }
        ],
    )

    # Printing the response
    print(response.content[0].text)
    
except Exception as e:
    print(e)

Error code: 400 - {'detail': "Guardrail 'Jar Jar Guardrail' blocked: Sanitization LLM returned invalid JSON."}
